---
# 触觉传感器负载实验分析
## 目的：判断是否需要软件滤波
---

**判断准则**：将噪声峰峰值(6σ)与最小可分辨信号变化量(ΔS_min)比较

| 比值 (6σ / ΔS_min) | 判断 |
|---|---|
| < 1/3 | 可不滤波 |
| 1/3 ~ 1 | 建议滤波 |
| > 1 | 必须滤波 |

## 1. 导入库与基础配置

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# 绘图配置
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# ============================================================
# 传感器与ADC参数
# ============================================================
FS = 200          # 采样频率 Hz
DT = 1/FS         # 采样周期 s
V_REF = 5.0       # 参考电压 V
ADC_BITS = 16     # ADC位数
LSB = V_REF / (2**15)  # LSB电压 (有符号16位)

# ============================================================
# 零负载噪声基准 (来自之前实验)
# ============================================================
NOISE_STD = 1.16  # 零负载噪声标准差 (LSB)
NOISE_6SIGMA = 6 * NOISE_STD  # 噪声峰峰值 ≈ 7 LSB

print(f"采样频率: {FS} Hz")
print(f"LSB电压: {LSB*1000:.4f} mV")
print(f"零负载噪声 σ: {NOISE_STD:.2f} LSB")
print(f"噪声峰峰值 6σ: {NOISE_6SIGMA:.1f} LSB")

## 2. 数据加载函数

In [ ]:
def load_tactile_data(file_path):
    """
    加载触觉传感器CSV数据
    """
    df = pd.read_csv(file_path)
    df.columns = ['Time_ms', 'RawValue', 'RawGround']
    df['Time_s'] = df['Time_ms'] / 1000
    df['Voltage_V'] = df['RawValue'] * V_REF / (2**15)
    return df

def detect_outliers(data, threshold=100):
    """检测异常值"""
    return np.abs(data) > threshold

def get_clean_data(df):
    """获取剔除异常值后的数据"""
    raw = df['RawValue'].values
    outliers = detect_outliers(raw, threshold=100)
    return raw[~outliers]

---
## 3. 加载实验数据
---

**实验设计建议**：
- 零负载：传感器空载状态，采集 >20s
- 最小负载：施加最小量程力(如0.01N)，保持稳定，采集 >20s
- (可选) 多级负载：0.01N, 0.05N, 0.1N, 0.5N 等

In [ ]:
# ============================================================
# === 请修改为你的实验文件路径 ===
# ============================================================

# 零负载数据 (用于噪声基准校验)
file_zero_load = r"请填入零负载数据文件路径.csv"

# 最小负载数据 (如 0.01N)
file_min_load = r"请填入最小负载数据文件路径.csv"

# 最小负载力值 (N)
FORCE_MIN = 0.01  # 请根据实际实验修改

# ============================================================
# 加载数据
# ============================================================
try:
    df_zero = load_tactile_data(file_zero_load)
    df_min = load_tactile_data(file_min_load)
    
    print("数据加载成功!")
    print(f"零负载数据点数: {len(df_zero)}, 时长: {df_zero['Time_s'].max():.1f}s")
    print(f"最小负载数据点数: {len(df_min)}, 时长: {df_min['Time_s'].max():.1f}s")
    DATA_LOADED = True
except FileNotFoundError as e:
    print(f"⚠️ 文件未找到: {e}")
    print("请修改上方的文件路径后重新运行此单元格")
    DATA_LOADED = False

---
## 4. 信号变化量分析
---

In [ ]:
if DATA_LOADED:
    # 获取干净数据
    raw_zero = get_clean_data(df_zero)
    raw_min = get_clean_data(df_min)
    
    # 计算统计量
    mean_zero = np.mean(raw_zero)
    mean_min = np.mean(raw_min)
    std_zero = np.std(raw_zero)
    std_min = np.std(raw_min)
    
    # 信号变化量 ΔS
    delta_S = abs(mean_min - mean_zero)
    
    print("=" * 60)
    print("【信号变化量分析】")
    print("=" * 60)
    print(f"零负载均值: {mean_zero:.2f} LSB")
    print(f"最小负载({FORCE_MIN}N)均值: {mean_min:.2f} LSB")
    print(f"信号变化量 ΔS: {delta_S:.2f} LSB")
    print(f"零负载噪声 σ: {std_zero:.2f} LSB")
    print(f"最小负载噪声 σ: {std_min:.2f} LSB")
else:
    print("⚠️ 请先加载数据")

## 5. 时域波形对比

In [ ]:
if DATA_LOADED:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 左图：时域波形对比
    ax1 = axes[0]
    n_show = min(1000, len(raw_zero), len(raw_min))
    t = np.arange(n_show) * DT * 1000
    
    ax1.plot(t, raw_zero[:n_show], 'b-', alpha=0.7, label=f'Zero Load (μ={mean_zero:.1f})', linewidth=0.8)
    ax1.plot(t, raw_min[:n_show], 'r-', alpha=0.7, label=f'Min Load {FORCE_MIN}N (μ={mean_min:.1f})', linewidth=0.8)
    ax1.axhline(y=mean_zero, color='b', linestyle='--', alpha=0.5)
    ax1.axhline(y=mean_min, color='r', linestyle='--', alpha=0.5)
    ax1.set_xlabel('Time (ms)')
    ax1.set_ylabel('ADC Value (LSB)')
    ax1.set_title('Time Domain Comparison')
    ax1.legend()
    
    # 右图：直方图对比
    ax2 = axes[1]
    ax2.hist(raw_zero, bins=50, alpha=0.6, label='Zero Load', color='blue', density=True)
    ax2.hist(raw_min, bins=50, alpha=0.6, label=f'Min Load {FORCE_MIN}N', color='red', density=True)
    ax2.axvline(x=mean_zero, color='b', linestyle='--', linewidth=2)
    ax2.axvline(x=mean_min, color='r', linestyle='--', linewidth=2)
    ax2.set_xlabel('ADC Value (LSB)')
    ax2.set_ylabel('Density')
    ax2.set_title('Distribution Comparison')
    ax2.legend()
    
    # 标注ΔS
    ax2.annotate('', xy=(mean_min, 0.3), xytext=(mean_zero, 0.3),
                arrowprops=dict(arrowstyle='<->', color='green', lw=2))
    ax2.text((mean_zero + mean_min)/2, 0.32, f'ΔS={delta_S:.1f}', 
            ha='center', fontsize=12, color='green', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ 请先加载数据")

---
## 6. 滤波必要性判断
---

In [ ]:
if DATA_LOADED:
    # 使用实测噪声
    noise_sigma = std_zero
    noise_6sigma = 6 * noise_sigma
    
    # 计算判断比值
    ratio = noise_6sigma / delta_S if delta_S > 0 else float('inf')
    
    print("=" * 60)
    print("【滤波必要性判断】")
    print("=" * 60)
    print(f"噪声标准差 σ: {noise_sigma:.2f} LSB")
    print(f"噪声峰峰值 6σ: {noise_6sigma:.1f} LSB")
    print(f"最小信号变化量 ΔS: {delta_S:.1f} LSB")
    print(f"判断比值 (6σ/ΔS): {ratio:.3f}")
    print("-" * 60)
    
    # 判断逻辑
    if ratio < 1/3:
        verdict = "✅ 可不滤波"
        reason = f"噪声峰峰值({noise_6sigma:.1f}) < ΔS/3({delta_S/3:.1f})，噪声影响可忽略"
        filter_recommend = False
    elif ratio < 1:
        verdict = "⚠️ 建议滤波"
        reason = f"噪声峰峰值({noise_6sigma:.1f})在ΔS的1/3~1倍之间，滤波可提升分辨率"
        filter_recommend = True
    else:
        verdict = "❌ 必须滤波"
        reason = f"噪声峰峰值({noise_6sigma:.1f}) > ΔS({delta_S:.1f})，噪声已淹没信号"
        filter_recommend = True
    
    print(f"\n判断结果: {verdict}")
    print(f"原因: {reason}")
else:
    print("⚠️ 请先加载数据")
    filter_recommend = None

## 7. 信噪比与分辨率分析

In [ ]:
if DATA_LOADED:
    # 计算信噪比
    SNR_dB = 20 * np.log10(delta_S / noise_sigma) if noise_sigma > 0 else float('inf')
    
    # 计算可分辨的力分辨率
    # 假设线性关系：ΔS 对应 FORCE_MIN
    # 3σ对应的力分辨率
    force_resolution_3sigma = (3 * noise_sigma / delta_S) * FORCE_MIN if delta_S > 0 else float('inf')
    
    print("=" * 60)
    print("【信噪比与分辨率】")
    print("=" * 60)
    print(f"信噪比 SNR: {SNR_dB:.1f} dB")
    print(f"力分辨率 (3σ): {force_resolution_3sigma*1000:.3f} mN")
    print(f"灵敏度: {delta_S/FORCE_MIN:.1f} LSB/N")
    
    # 判断分辨率是否满足需求
    print("-" * 60)
    if force_resolution_3sigma < FORCE_MIN:
        print(f"✅ 力分辨率({force_resolution_3sigma*1000:.2f}mN) < 最小量程({FORCE_MIN*1000:.0f}mN)")
        print("   传感器可分辨最小量程的力变化")
    else:
        print(f"❌ 力分辨率({force_resolution_3sigma*1000:.2f}mN) > 最小量程({FORCE_MIN*1000:.0f}mN)")
        print("   需要滤波提升分辨率")
else:
    print("⚠️ 请先加载数据")

---
## 8. 结论汇总
---

In [ ]:
if DATA_LOADED:
    print("=" * 60)
    print("【实验结论汇总】")
    print("=" * 60)
    print(f"""
┌────────────────────────────────────────────────────────────┐
│  测量参数                                                  │
├────────────────────────────────────────────────────────────┤
│  零负载均值:        {mean_zero:>10.2f} LSB                       │
│  最小负载均值:      {mean_min:>10.2f} LSB                       │
│  信号变化量 ΔS:     {delta_S:>10.2f} LSB                       │
│  噪声标准差 σ:      {noise_sigma:>10.2f} LSB                       │
│  噪声峰峰值 6σ:     {noise_6sigma:>10.1f} LSB                       │
├────────────────────────────────────────────────────────────┤
│  判断比值 6σ/ΔS:    {ratio:>10.3f}                              │
│  信噪比 SNR:        {SNR_dB:>10.1f} dB                         │
│  力分辨率(3σ):      {force_resolution_3sigma*1000:>10.3f} mN                       │
├────────────────────────────────────────────────────────────┤
│  最终建议:          {verdict}                            │
└────────────────────────────────────────────────────────────┘
""")
    
    if filter_recommend:
        print("\n【推荐滤波参数】")
        print("-" * 40)
        print("方案: EMA (指数移动平均)")
        print("参数: α = 0.3")
        print("实现: y[n] = 0.3×x[n] + 0.7×y[n-1]")
        print("预期效果: 噪声抑制~59%, 延迟~5ms")
    else:
        print("\n【无需滤波】")
        print("-" * 40)
        print("当前ADC性能足以满足应用需求")
        print("可直接使用原始数据进行控制")
else:
    print("⚠️ 请先加载数据后运行分析")

---
## 附录：多级负载实验 (可选)
---

如果需要进行完整的标定实验，可以加载多组负载数据进行线性度分析。

In [ ]:
# ============================================================
# 多级负载实验配置 (可选)
# ============================================================

# 取消下方注释并填入文件路径进行多级负载分析

# load_configs = [
#     {'force': 0.00, 'file': r"零负载文件路径.csv"},
#     {'force': 0.01, 'file': r"0.01N负载文件路径.csv"},
#     {'force': 0.05, 'file': r"0.05N负载文件路径.csv"},
#     {'force': 0.10, 'file': r"0.10N负载文件路径.csv"},
#     {'force': 0.50, 'file': r"0.50N负载文件路径.csv"},
# ]

# results = []
# for cfg in load_configs:
#     df = load_tactile_data(cfg['file'])
#     raw = get_clean_data(df)
#     results.append({
#         'force': cfg['force'],
#         'mean': np.mean(raw),
#         'std': np.std(raw)
#     })

# # 绘制标定曲线
# forces = [r['force'] for r in results]
# means = [r['mean'] for r in results]
# stds = [r['std'] for r in results]

# plt.figure(figsize=(10, 6))
# plt.errorbar(forces, means, yerr=stds, fmt='o-', capsize=5, capthick=2)
# plt.xlabel('Force (N)')
# plt.ylabel('ADC Value (LSB)')
# plt.title('Force-ADC Calibration Curve')
# plt.grid(True, alpha=0.3)
# plt.show()